<a href="https://colab.research.google.com/github/DubeyDivyansh/video_production_engine/blob/main/quick_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 AI Video & Voice Generation Platform (Google Colab T4)

Comprehensive end-to-end platform for realistic talking head and whole-body video generation, zero-shot voice cloning, and audio-driven animation on **Google Colab Free NVIDIA Tesla T4 GPU (16GB VRAM)**.

### Supported Engines:
1. **F5-TTS**: Zero-shot voice cloning and natural speech synthesis from reference audio.
2. **SadTalker**: 3D motion coefficients with GFPGAN face restoration.
3. **LatentSync**: Latent diffusion lip-sync retaining full body movement and gestures.

---
### 0. GPU Verification

In [ ]:
# Verify CUDA GPU hardware (Tesla T4, 15-16 GB VRAM)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

---
# 🚀 1-Click Video Production Engine (Modular Runner)

This cell clones your `video_production_engine` module from GitHub, imports the unified pipeline, and runs the entire Voice Cloning + VRAM Fence + Video Generation flow in clean Python code!

In [ ]:
# 1. Setup workspace & clone our GitHub production engine
!git clone https://github.com/DubeyDivyansh/video_production_engine.git /content/app 2>/dev/null || (cd /content/app && git pull)
%cd /content/app

# 2. Install production dependencies
!pip install -q -r requirements.txt

# 3. Add to Python path and import unified pipeline
import sys
if '/content/app' not in sys.path:
    sys.path.append('/content/app')

from src.pipeline import VideoProductionPipeline
from IPython.display import HTML, display
from base64 import b64encode

# 4. Download user media assets if not present
!wget -q -O /content/custom_face.jpeg https://files.catbox.moe/puyuek.jpeg
!wget -q -O /content/custom_screenshot.jpg https://files.catbox.moe/h6ddvp.jpg
!wget -q -O /content/custom_audio.wav https://files.catbox.moe/dvin9r.wav

# 5. Initialize pipeline with Google Drive persistence
pipeline = VideoProductionPipeline(use_drive=True)

# 6. Generate complete video from script and face image
output_video = pipeline.produce_video(
    face_image_path='/content/custom_face.jpeg',
    ref_audio_path='/content/custom_audio.wav',
    script_text='Hello! Welcome to our modular video production platform running entirely on Google Colab!',
    output_name='divyansh_avatar_production.mp4',
    engine='sadtalker'
)

# 7. Inline video playback
mp4 = open(output_video, 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
display(HTML(f'''
<video width=512 controls autoplay loop>
  <source src="{data_url}" type="video/mp4">
</video>
'''))

---
### 1. Google Drive Persistence Mount
Persist checkpoints and outputs to Google Drive to prevent re-downloading across sessions.

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/video_gen_platform'
    for d in ['checkpoints', 'inputs', 'outputs']:
        os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)
    print(f'[✓] Persistent workspace active at: {BASE_DIR}')
except Exception as e:
    print('[!] Drive mount notice:', e)

---
### 2. Voice Generation: F5-TTS Zero-Shot Voice Cloning
Clone your voice from reference audio (`custom_audio.wav`) and synthesize any custom script.

In [ ]:
# Install F5-TTS & dependencies
!pip install -q f5-tts vocos soundfile

import os
import torch
import soundfile as sf
from f5_tts.api import F5TTS
from IPython.display import Audio

# 1. Download reference speech if not present
if not os.path.exists('/content/custom_audio.wav'):
    !wget -q -O /content/custom_audio.wav https://files.catbox.moe/dvin9r.wav

# Slice clean 7.5s reference prompt for optimal F5-TTS conditioning
!ffmpeg -y -i /content/custom_audio.wav -t 7.5 /content/ref_audio_prompt.wav &> /dev/null

ref_audio = '/content/ref_audio_prompt.wav'
ref_text = "Hello everyone, first of all thanks for giving this opportunity."
target_text = "Hello! This is a test of our end to end video generation platform running on Google Colab!"
output_wav = '/content/f5_tts_speech.wav'

print('[*] Initializing F5-TTS on CUDA...')
f5 = F5TTS(device='cuda')

print('[*] Synthesizing cloned speech...')
wav, sr, _ = f5.infer(
    ref_file=ref_audio,
    ref_text=ref_text,
    gen_text=target_text
)

sf.write(output_wav, wav, sr)
print(f'[✓] Cloned speech generated: {output_wav}')
display(Audio(output_wav))

# Copy to Google Drive if mounted
!cp /content/f5_tts_speech.wav /content/drive/MyDrive/video_gen_platform/outputs/f5_tts_speech.wav 2>/dev/null || true

---
### 3. VRAM Memory Fence
Flushes GPU cache to ensure full 15GB VRAM availability for video rendering.

In [ ]:
import gc
import torch

if 'f5' in locals():
    del f5

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print('[✓] VRAM cache cleared.')
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

---
### 4. SadTalker 3D Motion & GFPGAN Face Animation
Generates high-resolution facial motion from single portrait photos.

In [ ]:
# Setup SadTalker with Noble Python 3.8 fix
%cd /content
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2 2>/dev/null || true
!apt-get install -y python3.8-dev ffmpeg 2>/dev/null || true

!git clone https://github.com/Winfredy/SadTalker.git 2>/dev/null || true
%cd /content/SadTalker

!bash scripts/download_models.sh 2>/dev/null || true
print('[✓] SadTalker models initialized!')

In [ ]:
# Download user media assets
!wget -q -O /content/face_passport.jpeg https://files.catbox.moe/puyuek.jpeg
!wget -q -O /content/face_screenshot.jpg https://files.catbox.moe/h6ddvp.jpg
!wget -q -O /content/speech_audio.wav https://files.catbox.moe/dvin9r.wav

%cd /content/SadTalker

# Generate video from Passport photo
!python3.8 inference.py --driven_audio /content/speech_audio.wav \
           --source_image /content/face_passport.jpeg \
           --result_dir ./results --still --preprocess full --enhancer gfpgan

# Generate video from Screenshot photo
!python3.8 inference.py --driven_audio /content/speech_audio.wav \
           --source_image /content/face_screenshot.jpg \
           --result_dir ./results --still --preprocess full --enhancer gfpgan

---
### 5. LatentSync Lip-Sync & Whole Body Motion (T4 Optimized)
Applies the Turing sm_75 FP16 patch and DeepCache to render whole-body videos without cropping gestures.

In [ ]:
%cd /content
!git clone https://github.com/bytedance/LatentSync.git 2>/dev/null || true
%cd /content/LatentSync

# Install dependencies with isolated flags
!pip install -q diffusers==0.32.2 transformers decord accelerate einops omegaconf opencv-python mediapipe librosa DeepCache

# Apply Tesla T4 Turing (sm_75) FP16 Patch
!python -c "import re; p='scripts/inference.py'; c=open(p).read(); open(p,'w').write(re.sub(r'capability\(\)\[0\]\s*>\s*7', 'capability()[0] >= 7', c))"
print('[✓] T4 FP16 patch applied to LatentSync!')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('[✓] Ready for LatentSync whole-body generation!')

---
### 6. Video Preview Player
Inline HTML5 player for previewing rendered animations.

In [ ]:
import glob
from IPython.display import HTML
from base64 import b64encode

results = sorted(glob.glob('/content/SadTalker/results/*/*.mp4'))
if results:
    latest_mp4 = results[-1]
    print('Previewing latest render:', latest_mp4)
    mp4 = open(latest_mp4, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'''
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
else:
    print('No generated MP4 files found in results.')